# Pull messages from a Pub/Sub subscription

Testing / inspecting messages already published to a topic. In Pub/Sub you consume from a **subscription** attached to the topic, not the topic itself. This does a **one-time synchronous pull** and does **not** acknowledge the messages, so they stay available (they redeliver after the ack deadline).

In [1]:
import json
import google.auth
from google.cloud import pubsub_v1

from google.cloud.pubsub_v1.types import ReceivedMessage, PullResponse

from rockyfilescom.wrapper import Session as RockyFilesCom

from files_sdk import File

# Authenticate as YOUR user identity via Application Default Credentials (ADC).
# If this errors or shows the wrong account, run this once in a terminal:
#     gcloud auth application-default login
#     gcloud auth application-default set-quota-project rmr-cloud-services
credentials, adc_project = google.auth.default()

PROJECT_ID = 'rmr-cloud-services'
subscriber = pubsub_v1.SubscriberClient(credentials=credentials)
print('ADC project:', adc_project)
print('Credential type:', type(credentials).__name__)

# --- Service account alternative (previous approach) ---
# import os
# from dotenv import load_dotenv
# from google.oauth2 import service_account
# load_dotenv('../../.env')
# credentials = service_account.Credentials.from_service_account_file(os.getenv('GCLOUD_CREDS_PATH'))
# subscriber = pubsub_v1.SubscriberClient(credentials=credentials)

ADC project: rmr-cloud-services
Credential type: Credentials


In [2]:
# TODO: set the subscription attached to your topic
SUBSCRIPTION_ID = 'filescom-events-sub'  # <-- fill this in

subscription_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)
print('Pulling from:', subscription_path)

Pulling from: projects/rmr-cloud-services/subscriptions/filescom-events-sub


### Don't know the subscription name?
Run the cell below to list every subscription in the project, then set `SUBSCRIPTION_ID` above.

In [3]:
project_path = f'projects/{PROJECT_ID}'
for sub in subscriber.list_subscriptions(request={'project': project_path}):
    print(sub.name, '->', sub.topic)

projects/rmr-cloud-services/subscriptions/filescom-events-sub -> projects/rmr-cloud-services/topics/filescom-events


In [4]:

# PROJECT_ID = 'rmr-cloud-services'
# SUBSCRIPTION_ID = 'filescom-events-sub'

subscription_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)

def get_messages(
    subscription: str = subscription_path,
    max_messages: int = 50,
    timeout: int = 30
) -> list[ReceivedMessage]:
    """Syncronous one-time pull. Does NOT ack so messages stay on the subscription.

    Args:
        subscription (str, optional): name of pubsub subscription. Defaults to subscription_path.
        max_messages (int, optional): maximum number of messages to pull. Defaults to 50.
        timeout (int, optional): Defaults to 30.

    Returns:
        list[ReceivedMessage]: messages from pubsub service
    """    
    response = subscriber.pull(
        request={'subscription': subscription, 'max_messages': max_messages},
        timeout=timeout,
    )

    received: list[ReceivedMessage] = response.received_messages

    # NOTE: not calling subscriber.acknowledge(...), so these redeliver after the
    # ack deadline. To remove them once you're done, uncomment:
    # ack_ids = [rm.ack_id for rm in received]
    # if ack_ids:
    #     subscriber.acknowledge(request={'subscription': subscription_path, 'ack_ids': ack_ids})
    #     print('Acknowledged', len(ack_ids), 'message(s)')

    num_messages = len(received)
    print(f'Got {num_messages} message{'s' if num_messages > 1 else ''}\n')
    return received

In [5]:
# wtf, i get a different number of messages every time i call this
# - this is because messages are 'leased' out to whoever pulls them
# - messages that are pulled are pretty much invisible to other pullers for the length of the ack-deadline (default 10 seconds) 
messages = get_messages()

Got 4 messages



In [ ]:
messages[9]
message_data = json.loads(messages[9].message.data)
path = message_data.get("path")

rfc_file = rfc.get_file(path)

In [ ]:
path = rfc_file.path
# print(rfc_file.display_name)

index = 


In [ ]:
for k, v in rfc_file.__dict__.items():
    print(f"{k}: {v}")

In [ ]:
# utils

from rockyclickup.wrapper import Session as rcu_session
from rockyclickup.models import DataFile



rcu = rcu_session()


def get_datafile(rfc_file: File):
    # lookup datafile on clickup by name/path 

    rfc_name = rfc_file.display_name
    rfc_path = rfc_file.path

    search_res = rcu.query(DataFile).equals("ftp_filename").equals("ftp_directory").all()
    search_res = rcu.







    # return found datafile or None if not found

    pass



In [ ]:
## message handlers

# file handlers

def handle_file_create(message_data: dict, rfc_file: File):
    print(f"handling file {'create:':<8} {message_data['path']}")
    

    # check if datafile already exists
    existing_file = get_datafile(rfc_file)
    if existing_file is not None:
        update_datafile()

    print(rfc_file)


    print(json.dumps(message_data, indent=4))


def handle_file_update(message_data: dict, rfc_file: File):
    print(f"handling file {'update:':<8} {message_data['path']}")

    print(json.dumps(message_data, indent=4))
    
    

def handle_file_read(message_data: dict, rfc_file: File):
    print(f"handling file {'read:':<8} {message_data['path']}")

    print(json.dumps(message_data, indent=4))
    


def handle_file_move(message_data: dict, rfc_file: File):
    print(f"handling file {'move:':<8} {message_data['path']}")

    print(json.dumps(message_data, indent=4))
    
    


def handle_file_destroy(message_data: dict, rfc_file: File):
    print(f"handling file {'destroy:':<8} {message_data['path']}")

    print(json.dumps(message_data, indent=4))
    
    


# directory handlers

def handle_dir_create(message_data: dict, rfc_file: File):
    print(f"handling directory {'create:':<8} {message_data['path']}")

    print(json.dumps(message_data, indent=4))


def handle_dir_update(message_data: dict, rfc_file: File):
    print(f"handling directory {'update:':<8} {message_data['path']}")
    
    print(json.dumps(message_data, indent=4))


def handle_dir_read(message_data: dict, rfc_file: File):
    print(f"handling directory {'read:':<8} {message_data['path']}")
    
    print(json.dumps(message_data, indent=4))


def handle_dir_move(message_data: dict, rfc_file: File):
    print(f"handling directory {'move:':<8} {message_data['path']}")
    
    print(json.dumps(message_data, indent=4))


def handle_dir_destroy(message_data: dict, rfc_file: File):
    print(f"handling directory {'destroy:':<8} {message_data['path']}")
    
    print(json.dumps(message_data, indent=4))


In [ ]:
"""
a ReceivedMessage looks like this:

    ack_id: "LF1GSFE3GQhoUQ5PXiM_NSAoRRoLUxNRXE8S...",
    message {
        data: {
            "default": {
                "source":"Files.com"
            },
            "action":"create",
            "interface":"desktop",
            "path":"Clients/Orange Tree Co-RMROTREE/rmrgreen.png",
            "destination":"Clients/Orange Tree Co-RMROTREE/rmrgreen.png",
            "at":"2026-07-29T15:30:19-04:00",
            "username":"james.richmond@rmrbenefits.com",
            "ip":"73.20.57.233",
            "type":"file",
            "size":0
        },
        message_id: "20844211737740070"
        publish_time {
            seconds: 1785353420,
            nanos: 273000000
        }
    }
    
"""


func_map = {
    "file": {
        "create": handle_file_create,
        "update": handle_file_update,
        "read": handle_file_read,
        "move": handle_file_move,
        "destroy": handle_file_destroy,
    },
    "directory": {
        "create": handle_dir_create,
        "update": handle_dir_update,
        "read": handle_dir_read,
        "move": handle_dir_move,
        "destroy": handle_dir_destroy,
    }
}

rfc = RockyFilesCom() # needs env variable `FILESCOM_KEY`
    
def handle_message(received_message: ReceivedMessage):

    # open message data
    message_data = json.loads(received_message.message.data)
    path = message_data.get("path")
    type = message_data.get("type")
    action = message_data.get("action")

    # get file data from filescom
    rfc_file = rfc.get_file(path)

    # get function based on `type` and `action`
    func = func_map.get(type, {}).get(action, None)
    if func:
        return func(message_data, rfc_file)

    raise ValueError(f"* Unhandled action: {type}.{action} *")




In [ ]:
data = json.loads(messages[9].message.data)
path_2 = "\\" + data['path'].replace("/", "\\")
rfc_file = rfc.get_file(path_2)


In [ ]:
rfc_file.custom_metadata

In [ ]:
messages = get_messages()

In [ ]:
data = json.loads(messages[9].message.data)
print(f"{data.get("path")} ({data.get("type")} {data.get("action")})")

In [ ]:
handled = handle_message(messages[9])

In [ ]:
for message in messages:
    handle_message(message)
    # print()
    break